# 01. UCI Online Retail — 탐색적 데이터 분석 (EDA)

UCI Online Retail II 데이터셋(2010-2011)을 활용한 이커머스 거래 데이터 탐색적 분석입니다.

**데이터셋**: UK 기반 온라인 리테일러의 실제 거래 기록
- 기간: 2010-12-01 ~ 2011-12-09
- 규모: ~541,909 거래, 4,372 고객
- 주요 특이점: CustomerID 결측 ~25%, 음수 Quantity(반품)

**분석 목표**:
1. 데이터 품질 파악 (결측, 이상치)
2. 매출 구조 이해 (국가별, 월별, 상품별)
3. 통계적 검정으로 비즈니스 가설 검증

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

print('Libraries loaded.')

## 1. 데이터 로딩 및 기본 정보

In [ ]:
df = pd.read_csv('../data/online_retail.csv', encoding='utf-8')

# InvoiceDate를 datetime으로 변환
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f'Shape: {df.shape}')
print(f'Period: {df["InvoiceDate"].min()} ~ {df["InvoiceDate"].max()}')
print(f'\nColumns: {list(df.columns)}')
df.head()

In [ ]:
print('=== 데이터 타입 ===')
print(df.dtypes)
print(f'\n=== 기본 통계 ===')
df.describe()

## 2. 결측값 분석

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)

print('=== 결측값 현황 ===')
print(missing_df)
print(f'\n전체 행 수: {len(df):,}')
print(f'CustomerID 결측 행: {df["Customer ID"].isnull().sum() if "Customer ID" in df.columns else df["CustomerID"].isnull().sum():,}')

In [ ]:
# 결측값 시각화
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e15759' if p > 0 else '#4e79a7' for p in missing_pct]
missing_pct.plot(kind='bar', ax=ax, color=colors)
ax.set_title('Missing Values by Column (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Missing %')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

for i, v in enumerate(missing_pct):
    if v > 0:
        ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. 이상치 탐지

In [ ]:
# CustomerID 컬럼명 통일 (데이터셋 버전에 따라 다를 수 있음)
if 'Customer ID' in df.columns:
    df.rename(columns={'Customer ID': 'CustomerID'}, inplace=True)

# 음수 Quantity = 반품/취소 거래
returns = df[df['Quantity'] < 0]
normal = df[df['Quantity'] > 0]

print(f'정상 거래: {len(normal):,} ({len(normal)/len(df)*100:.1f}%)')
print(f'반품/취소: {len(returns):,} ({len(returns)/len(df)*100:.1f}%)')
print(f'Quantity=0: {(df["Quantity"]==0).sum():,}')

# 극단적 UnitPrice
print(f'\nUnitPrice 통계:')
print(f'  Min: {df["Price"].min() if "Price" in df.columns else df["UnitPrice"].min()}')
print(f'  Max: {df["Price"].max() if "Price" in df.columns else df["UnitPrice"].max()}')
print(f'  Median: {df["Price"].median() if "Price" in df.columns else df["UnitPrice"].median()}')

In [ ]:
# UnitPrice/Price 컬럼명 통일
price_col = 'Price' if 'Price' in df.columns else 'UnitPrice'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quantity 분포 (이상치 제거 후)
qty_clean = df[(df['Quantity'] > 0) & (df['Quantity'] < df['Quantity'].quantile(0.99))]
axes[0].hist(qty_clean['Quantity'], bins=50, color='#4e79a7', edgecolor='white', alpha=0.8)
axes[0].set_title('Quantity Distribution (< 99th percentile)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Quantity')
axes[0].set_ylabel('Frequency')
axes[0].axvline(qty_clean['Quantity'].median(), color='#e15759', linestyle='--',
                label=f'Median: {qty_clean["Quantity"].median():.0f}')
axes[0].legend()

# UnitPrice 분포
price_clean = df[(df[price_col] > 0) & (df[price_col] < df[price_col].quantile(0.99))]
axes[1].hist(price_clean[price_col], bins=50, color='#f28e2b', edgecolor='white', alpha=0.8)
axes[1].set_title(f'{price_col} Distribution (< 99th percentile)', fontsize=13, fontweight='bold')
axes[1].set_xlabel(price_col)
axes[1].set_ylabel('Frequency')
axes[1].axvline(price_clean[price_col].median(), color='#e15759', linestyle='--',
                label=f'Median: {price_clean[price_col].median():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. 매출 파생 변수 생성 및 데이터 정제

In [ ]:
price_col = 'Price' if 'Price' in df.columns else 'UnitPrice'

# Revenue 컬럼 생성
df['Revenue'] = df['Quantity'] * df[price_col]

# 분석용 정제 데이터 (반품 제외, CustomerID 존재)
df_clean = df[
    (df['Quantity'] > 0) &
    (df[price_col] > 0) &
    (df['CustomerID'].notna())
].copy()

print(f'원본: {len(df):,} rows')
print(f'정제 후: {len(df_clean):,} rows ({len(df_clean)/len(df)*100:.1f}%)')
print(f'고유 고객 수: {df_clean["CustomerID"].nunique():,}')
print(f'고유 상품 수: {df_clean["StockCode"].nunique():,}')
print(f'총 매출: £{df_clean["Revenue"].sum():,.0f}')

## 5. 국가별 매출 분석

In [ ]:
country_rev = df_clean.groupby('Country').agg(
    Revenue=('Revenue', 'sum'),
    Orders=('Invoice' if 'Invoice' in df_clean.columns else 'InvoiceNo', 'nunique'),
    Customers=('CustomerID', 'nunique')
).sort_values('Revenue', ascending=False)

country_rev['Revenue_Share'] = (country_rev['Revenue'] / country_rev['Revenue'].sum() * 100).round(2)

print('=== Top 10 Countries by Revenue ===')
country_rev.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 국가 매출
top10 = country_rev.head(10)
colors = ['#e15759'] + ['#4e79a7'] * 9  # UK 강조
bars = axes[0].barh(top10.index[::-1], top10['Revenue'][::-1], color=colors[::-1])
axes[0].set_xlabel('Revenue (£)')
axes[0].set_title('Top 10 Countries by Revenue', fontsize=14, fontweight='bold')

for bar, rev in zip(bars, top10['Revenue'][::-1]):
    axes[0].text(bar.get_width() + top10['Revenue'].max() * 0.01,
                 bar.get_y() + bar.get_height()/2,
                 f'£{rev:,.0f}', va='center', fontsize=9)

# UK vs Non-UK 비율
uk_rev = country_rev.loc['United Kingdom', 'Revenue']
non_uk_rev = country_rev['Revenue'].sum() - uk_rev
axes[1].pie(
    [uk_rev, non_uk_rev],
    labels=['UK', 'Non-UK'],
    autopct='%1.1f%%',
    colors=['#e15759', '#4e79a7'],
    startangle=90,
    textprops={'fontsize': 12, 'fontweight': 'bold'}
)
axes[1].set_title('Revenue: UK vs Non-UK', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

uk_pct = uk_rev / (uk_rev + non_uk_rev) * 100
print(f'UK 매출 비중: {uk_pct:.1f}%')

## 6. 월별 매출 추이 및 계절성

In [ ]:
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')

monthly = df_clean.groupby('YearMonth').agg(
    Revenue=('Revenue', 'sum'),
    Orders=('Invoice' if 'Invoice' in df_clean.columns else 'InvoiceNo', 'nunique'),
    Customers=('CustomerID', 'nunique'),
    AvgOrderValue=('Revenue', 'mean')
).reset_index()

monthly['YearMonth_str'] = monthly['YearMonth'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# 월별 매출
bars = axes[0].bar(monthly['YearMonth_str'], monthly['Revenue'],
                    color='#4e79a7', edgecolor='white')

# 11월 (최대 매출) 하이라이트
max_idx = monthly['Revenue'].idxmax()
bars[max_idx].set_color('#e15759')
axes[0].annotate(
    f'Peak: £{monthly.loc[max_idx, "Revenue"]:,.0f}',
    xy=(max_idx, monthly.loc[max_idx, 'Revenue']),
    xytext=(max_idx - 2, monthly.loc[max_idx, 'Revenue'] * 1.1),
    arrowprops=dict(arrowstyle='->', color='#e15759', lw=2),
    fontsize=11, fontweight='bold', color='#e15759'
)

axes[0].set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Revenue (£)')
axes[0].tick_params(axis='x', rotation=45)

# 월별 고객 수 + 주문 수
axes[1].plot(monthly['YearMonth_str'], monthly['Customers'],
             marker='o', linewidth=2, label='Unique Customers', color='#4e79a7')
ax2 = axes[1].twinx()
ax2.plot(monthly['YearMonth_str'], monthly['Orders'],
         marker='s', linewidth=2, label='Orders', color='#f28e2b', linestyle='--')

axes[1].set_title('Monthly Customers & Orders', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Unique Customers', color='#4e79a7')
ax2.set_ylabel('Orders', color='#f28e2b')
axes[1].tick_params(axis='x', rotation=45)

lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

## 7. Top 10 상품 분석

In [ ]:
product_rev = df_clean.groupby(['StockCode', 'Description']).agg(
    Revenue=('Revenue', 'sum'),
    Quantity=('Quantity', 'sum'),
    Orders=('Invoice' if 'Invoice' in df_clean.columns else 'InvoiceNo', 'nunique')
).sort_values('Revenue', ascending=False).head(10)

print('=== Top 10 Products by Revenue ===')
product_rev

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

labels = [f'{code}\n{desc[:25]}...' if len(desc) > 25 else f'{code}\n{desc}'
          for (code, desc) in product_rev.index]

bars = ax.barh(labels[::-1], product_rev['Revenue'].values[::-1],
               color=sns.color_palette('colorblind', 10)[::-1])

for bar, rev in zip(bars, product_rev['Revenue'].values[::-1]):
    ax.text(bar.get_width() + product_rev['Revenue'].max() * 0.01,
            bar.get_y() + bar.get_height()/2,
            f'£{rev:,.0f}', va='center', fontsize=9)

ax.set_xlabel('Revenue (£)')
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. 통계적 검정

### 8-1. Independent t-test: UK vs Non-UK 평균 주문 금액 (AOV)

In [ ]:
invoice_col = 'Invoice' if 'Invoice' in df_clean.columns else 'InvoiceNo'

# 주문 단위 매출 집계
order_rev = df_clean.groupby([invoice_col, 'Country'])['Revenue'].sum().reset_index()
order_rev['is_UK'] = order_rev['Country'] == 'United Kingdom'

uk_aov = order_rev[order_rev['is_UK']]['Revenue']
non_uk_aov = order_rev[~order_rev['is_UK']]['Revenue']

# Independent samples t-test (Welch's)
t_stat, p_value = stats.ttest_ind(uk_aov, non_uk_aov, equal_var=False)

print('=== Independent t-test: UK vs Non-UK AOV ===')
print(f'UK AOV: £{uk_aov.mean():.2f} (n={len(uk_aov):,})')
print(f'Non-UK AOV: £{non_uk_aov.mean():.2f} (n={len(non_uk_aov):,})')
print(f'\nt-statistic: {t_stat:.4f}')
print(f'p-value: {p_value:.6f}')
print(f'\nConclusion: {"Statistically significant (p < 0.05)" if p_value < 0.05 else "Not statistically significant"}')
print(f'→ UK와 Non-UK 고객의 평균 주문 금액 차이는 {"통계적으로 유의미" if p_value < 0.05 else "유의미하지 않"}합니다.')

### 8-2. Chi-square test: 국가별 반품률 차이

In [ ]:
# UK vs Non-UK 반품 여부 교차표
df['is_UK'] = df['Country'] == 'United Kingdom'
df['is_return'] = df['Quantity'] < 0

contingency = pd.crosstab(df['is_UK'], df['is_return'])
print('=== Contingency Table ===')
print(contingency)

chi2, p_value_chi, dof, expected = stats.chi2_contingency(contingency)

# Cramer's V (effect size)
n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print(f'\n=== Chi-square Test: Return Rate by Country (UK vs Non-UK) ===')
print(f'Chi-square statistic: {chi2:.4f}')
print(f'p-value: {p_value_chi:.6f}')
print(f'Degrees of freedom: {dof}')
print(f"Cramer's V (effect size): {cramers_v:.4f}")

uk_return_rate = contingency.loc[True, True] / contingency.loc[True].sum() * 100
non_uk_return_rate = contingency.loc[False, True] / contingency.loc[False].sum() * 100

print(f'\nUK Return Rate: {uk_return_rate:.2f}%')
print(f'Non-UK Return Rate: {non_uk_return_rate:.2f}%')
print(f'\nConclusion: {"Statistically significant (p < 0.05)" if p_value_chi < 0.05 else "Not statistically significant"}')
print(f'→ UK와 Non-UK 간 반품률 차이는 {"통계적으로 유의미" if p_value_chi < 0.05 else "유의미하지 않"}합니다. (Cramer\'s V={cramers_v:.3f})')

## 9. Key Findings 요약

| # | 발견 | 수치 | 비즈니스 의미 |
|---|------|------|---------------|
| 1 | CustomerID 결측 ~25% | 데이터 품질 이슈 | 비회원 구매 추적 불가 → 회원가입 유도 필요 |
| 2 | UK 매출 비중 ~82% | 단일 시장 의존 | 해외 시장 확장 기회 존재 |
| 3 | 11월 매출 피크 | 계절적 패턴 | 홀리데이 시즌 재고/마케팅 집중 필요 |
| 4 | 반품률 국가별 차이 | 통계적 유의미 | 국가별 반품 정책 차별화 검토 |
| 5 | 소수 상품 매출 집중 | 파레토 패턴 | Top SKU 재고 관리 우선순위화 |

→ 다음 노트북(02_rfm_segmentation)에서 고객 세그멘테이션으로 심화 분석합니다.